# 第 18 章　重回帰と特徴量

前章の直線は時刻が 1 進むたびに，同じ幅だけ予測を上げました。朝と夕方に山があるなら，どうすれば同じ時刻の情報を使って，二つの山を表せるでしょうか。さらに，土日・祝日かどうかによって，朝と昼のどちらが多くなるかを見分けられるでしょうか。

前半では，本文に合わせて予測に渡す列を作ります。後半の約 20 分では，時刻の列に日種別を一列加え，自分が選んだ時間帯の予測を比べます。

Colab では「ドライブにコピーを保存」で自分用のノートを作ります。CSV が手元にない場合は公開 URL から読み込みます。ローカルでは，配布 ZIP の配置のまま開くか，ノートと CSV を同じフォルダーに置いて実行できます。

## 本文と一緒に実行

### 予測と評価の条件

前日終了時点に，翌日の各時間帯の貸出件数を予測します。2011 年で規則を決め，2012 年 1〜6 月の収録された同じ検証行で比べます。2012 年後半は秋の候補比較から保留します。前章でも同じ検証行を使っているので，ここでは，列の作り方を変えた候補の違いを同じ記録で読みます。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder

file_name = "bike-sharing-hourly.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

hourly = pd.read_csv(csv_path, parse_dates=["dteday"])
date = hourly["dteday"]
train = hourly.loc[date < "2012-01-01"]
valid = hourly.loc[(date >= "2012-01-01") & (date < "2012-07-01")]
y_train = train["cnt"]
plt.rcParams["figure.figsize"] = (7.2, 3.8)
print(f"学習 {len(train):,} 行，検証 {len(valid):,} 行")

### 時刻を目印にする

8 時台を数値で表すと `hr = 8` です。モデルはこの数に同じ傾きを掛けて予測します。8 時台を区別する目印を作るなら，`hr_8` の値は 8 時台だけ 1 になり，この列の係数は 8 時台の予測だけに加わります。

下のセルでは，学習側の `hr` をもとに `fit_transform()` で目印の列を作ります。検証側では，同じ列に 0・1 を入れる `transform()` だけを呼びます。行数は変わらず，0 時台を基準にして両方の表に 23 列できます。三つの時刻を変換した小表で，`hr_8` と `hr_13` がどこで 1 になるかを確かめてください。

In [ ]:
hour_encoder = OneHotEncoder(drop="first", sparse_output=False)
hour_encoder.set_output(transform="pandas")
X_hour_train = hour_encoder.fit_transform(train[["hr"]])
X_hour_valid = hour_encoder.transform(valid[["hr"]])

print(X_hour_train.shape, X_hour_valid.shape)

In [ ]:
example_hours = pd.DataFrame({"hr": [0, 8, 13]})
example_flags = hour_encoder.transform(example_hours)
example_hours.join(example_flags[["hr_8", "hr_13"]]).astype(int)

### 時刻ごとの予測

回帰を学習して，切片と各列の係数を読みます。0 時台はすべての目印が 0 です。8 時台なら，切片とどの係数を足すと予測値になるでしょうか。

次の表では，時刻ごとの学習平均とモデルの予測値を照合します。各時刻を表す目印が別なので，ある時刻の予測値を動かしても，別の時刻の誤差は変わりません。各時刻の学習平均が，その時刻の残差の二乗和を最小にします。

図では，直線の予測，目印を使う予測，検証の実測平均のうち，朝と夕方に山がある線を探します。同じ 8 時台であれば，どちらの日種別でも目印は同じです。同じ時刻でも日によって貸出が変わるとき，どんな列が手掛かりになるでしょうか。

In [ ]:
hour_model = LinearRegression()
hour_model.fit(X_hour_train, y_train)
hour_prediction = hour_model.predict(X_hour_valid)

hour_coef = pd.Series(hour_model.coef_, index=X_hour_train.columns)
print(f"切片: {hour_model.intercept_:.2f}")
hour_coef.loc[["hr_8", "hr_13", "hr_17"]].round(2)

In [ ]:
hour_grid = pd.DataFrame({"hr": range(24)})
by_hour = train.groupby("hr")["cnt"].mean().rename("train_mean").to_frame()
by_hour["hour_model"] = hour_model.predict(hour_encoder.transform(hour_grid))
by_hour.loc[[0, 8, 13, 17]].round(2)

In [ ]:
line_model = LinearRegression().fit(train[["hr"]], y_train)
observed_by_hour = valid.groupby("hr")["cnt"].mean()
fig, ax = plt.subplots(layout="constrained")
ax.plot(hour_grid["hr"], line_model.predict(hour_grid), color="#6d7780",
        linestyle=":", label="Numeric hour")
ax.plot(by_hour.index, by_hour["hour_model"], color="#195b8a",
        marker="o", markersize=3.5, label="Categorical hour")
ax.plot(observed_by_hour.index, observed_by_hour, color="#a7421f",
        marker="s", markersize=3.5, linestyle="--", label="Observed mean")
ax.set(xlabel="Hour of day", ylabel="Rentals per hour",
       xticks=[0, 4, 8, 13, 17, 23], ylim=(0, 600))
ax.legend(frameon=False, loc="upper left")
plt.show()

### 日種別を加える

`workingday = 0` は土日・祝日，`workingday = 1` はそれ以外の日です。時刻の目印の横にこの列を足し，係数を求め直します。同じ時刻では，1 の日に 0 の日より何件多く予測するかを，係数から読んでください。8 時台での二つの日種別の予測の差と，13 時台での差はどうなるでしょうか。

In [ ]:
X_add_train = X_hour_train.assign(workingday=train["workingday"])
X_add_valid = X_hour_valid.assign(workingday=valid["workingday"])

add_model = LinearRegression()
add_model.fit(X_add_train, y_train)
add_prediction = add_model.predict(X_add_valid)

add_coef = pd.Series(add_model.coef_, index=X_add_train.columns)
print(f"切片: {add_model.intercept_:.2f}")
add_coef.loc[["hr_8", "hr_13", "workingday"]].round(2)

### 時刻と日種別の組合せ

8 時台は土日・祝日以外の日の方が多く，13 時台は土日・祝日の方が多いとします。このとき，加算モデルのように，どの時刻でも日種別による差が同じであれば，朝と昼で大小を入れ替えられるでしょうか。

「8 時台で土日・祝日」を `08_0`，「8 時台で土日・祝日以外」を `08_1` とする組合せのラベルを用意します。同じ時刻でも二つのラベルは別々の列を使うので，予測の差を時刻別に変えられます。このように一つの特徴の違いが他の条件によって変わることを交互作用と呼びます。

二つ目のセルでは，学習側で組合せの列を決め，検証側に同じ変換を使います。0 時台の土日・祝日（`00_0`）が基準になります。24 時刻と二つの日種別の組がすべて学習側にあったとき，変換後の列数はいくつでしょうか。

In [ ]:
hour_day = hourly[["hr", "workingday"]].copy()
hour_text = hour_day["hr"].astype(str).str.zfill(2)
day_text = hour_day["workingday"].astype(str)
hour_day["hour_day"] = hour_text + "_" + day_text
hour_day.head(3)

In [ ]:
joint_encoder = OneHotEncoder(drop="first", sparse_output=False)
joint_encoder.set_output(transform="pandas")
X_joint_train = joint_encoder.fit_transform(hour_day.loc[train.index, ["hour_day"]])
X_joint_valid = joint_encoder.transform(hour_day.loc[valid.index, ["hour_day"]])

joint_model = LinearRegression()
joint_model.fit(X_joint_train, y_train)
joint_prediction = joint_model.predict(X_joint_valid)
print(X_joint_train.shape, X_joint_valid.shape)

### 二つの日種別の形

上段が検証期間の実測平均，中段が加算モデル，下段が組合せモデルです。横軸の 8 と 13 を見て，実測平均ではどちらの日種別の線が上かを確かめます。加算モデルと組合せモデルのうち，この大小の入れ替わりを表せるのはどちらでしょうか。形を表せても，実測平均までの高さにどれだけ差があるかを読みます。

In [ ]:
scored = valid.assign(
    constant=y_train.mean(),
    line=line_model.predict(valid[["hr"]]),
    hour_only=hour_prediction,
    additive=add_prediction,
    joint=joint_prediction,
)
curve_columns = ["cnt", "additive", "joint"]
by_condition = scored.groupby(["hr", "workingday"])[curve_columns].mean()
by_condition.loc[[8, 13, 17]].round(2)

In [ ]:
fig, axes = plt.subplots(3, 1, sharex=True, sharey=True, figsize=(7.2, 7.1), layout="constrained")
for ax, column, title in zip(
    axes, curve_columns, ["Observed mean", "Additive model", "Joint model"],
):
    for day_type, color, marker, style, label in [
        (0, "#a7421f", "o", "--", "Weekend / holiday"),
        (1, "#195b8a", "s", "-", "Working day"),
    ]:
        values = by_condition.xs(day_type, level="workingday")
        ax.plot(values.index, values[column], color=color, marker=marker,
                linestyle=style, markersize=3.3, label=label)
    ax.set(title=title, ylabel="Rentals per hour", ylim=(0, 660))
axes[0].legend(frameon=False, loc="upper left")
axes[-1].set(xlabel="Hour of day", xticks=[0, 4, 8, 13, 17, 23])
plt.show()

### 誤差を比べる

図では時刻・日種別ごとに実測を平均しました。今度は検証の一行ずつについて，実測と予測の絶対誤差を計算してから平均します。全体の MAE はどの候補で最も小さいでしょうか。その候補で，どの時刻や日種別でも誤差が減るでしょうか。

最後の図は，組合せモデルの MAE から加算モデルの MAE を引いています。0 より下なら，その条件では組合せの方が実測に近くなりました。8 時台，13 時台，17 時台の表の数値と図の位置を照らして読んでください。

In [ ]:
prediction_columns = ["constant", "line", "hour_only", "additive", "joint"]
abs_errors = pd.DataFrame(index=scored.index)
for column in prediction_columns:
    abs_errors[column] = (scored["cnt"] - scored[column]).abs()

abs_errors.mean().rename("MAE").round(2)

In [ ]:
error_groups = abs_errors.groupby([scored["hr"], scored["workingday"]])
mae_by_condition = error_groups.mean()
mae_by_condition.insert(0, "rows", error_groups.size())
mae_by_condition.loc[[8, 13, 17], ["rows", "hour_only", "additive", "joint"]].round(2)

In [ ]:
change = mae_by_condition["joint"] - mae_by_condition["additive"]
fig, ax = plt.subplots(layout="constrained")
for day_type, color, marker, style, label in [
    (0, "#a7421f", "o", "--", "Weekend / holiday"),
    (1, "#195b8a", "s", "-", "Working day"),
]:
    values = change.xs(day_type, level="workingday")
    ax.plot(values.index, values, color=color, marker=marker, linestyle=style,
            markersize=3.5, label=label)
ax.axhline(0, color="#343a42", linewidth=0.9)
ax.set(xlabel="Hour of day", ylabel="Change in MAE (rentals per hour)",
       xticks=[0, 4, 8, 13, 17, 23], ylim=(-100, 55))
ax.legend(frameon=False, loc="lower right")
plt.show()

## 実習（約 20 分）

### 1. 着目する時間帯を選ぶ（4 分）

朝（7〜9 時台），昼（12〜14 時台），夕方（16〜18 時台）のうち一つを選びます。時刻だけのモデルへ日種別を一列加えると，その時間帯ではどちらの日種別の誤差が変わりそうでしょうか。春の図や講義で見た一日の形から，予想の理由を一文で書いてください。

**選んだ時間帯と予想**　…

### 2. 一列加えて比べる（8 分）

`X_practice_train` と `X_practice_valid` の右辺の `None` を，それぞれ時刻の目印に `workingday` を一列追加した表へ替えます。`X_hour_train` と `X_hour_valid` を使い，それぞれ `assign()` を呼びます。学習側の列は `train` から，検証側の列は `valid` から取り出してください。

`hour_range` の右辺には選んだ時間帯の両端を，例えば 10〜11 時台なら `(10, 11)` と書きます。二つの日種別について，対象行数，時刻のみ・加算・組合せの MAE を確認します。`workingday` の係数も表示し，これがどの時間帯の予測差になるのかを考えてください。

In [ ]:
X_practice_train = None  # 学習側の時刻の目印に日種別を加えた表
X_practice_valid = None  # 検証側の時刻の目印に日種別を加えた表
hour_range = None  # (最初の時刻, 最後の時刻) に置き換える

if X_practice_train is None or X_practice_valid is None or hour_range is None:
    print("学習・検証の入力表と時間帯を指定してください。")
else:
    practice_model = LinearRegression().fit(X_practice_train, y_train)
    practice_prediction = practice_model.predict(X_practice_valid)
    practice_errors = abs_errors[["hour_only", "joint"]].copy()
    practice_errors["with_day_type"] = (scored["cnt"] - practice_prediction).abs()

    start_hour, end_hour = hour_range
    selected_hours = scored["hr"].between(start_hour, end_hour)
    groups = practice_errors.loc[selected_hours].groupby(scored["workingday"])
    practice_summary = groups.mean()[["hour_only", "with_day_type", "joint"]]
    practice_summary.insert(0, "rows", groups.size())

    print(f"日種別の係数: {practice_model.coef_[-1]:.2f} 件")
    display(practice_summary.round(2))

### 3. 比較の意味を説明する（8 分）

選んだ時間帯で，時刻だけのモデルから日種別を加算すると MAE はどう変わりましたか。予想と比べ，その時間帯の日種別の違いを一つの係数で表すときに残る疑問を示します。組合せモデルではその疑問にどう応えられたかを，片方または両方の日種別の行数と MAE を添えて 3〜4 文で書いてください。最後に，別の時期や時間帯で何をそろえて比べたいかを一つ挙げます。

**結果と，次に確かめたいこと**　…

提出する場合は，報告に使った時間帯をセルに指定し，カーネルを再起動して上から全セルを実行します。選んだ時間帯，集計表，短い説明が残る状態でノートを保存してください。

### データの出典

Capital Bikeshare の時間別集計は，Hadi Fanaee-T が UCI Machine Learning Repository で公開した `hour.csv` のファイル名と改行コードを変更したものです。 [データ DOI](https://doi.org/10.24432/C5W894) ・ [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)